# Airline Satisfaction — CatBoost baseline

This notebook trains a binary classifier for `satisfaction`, evaluates it with **ROC AUC** on a stratified holdout, then fits on all labeled rows and writes `submission_catboost.csv`. It assumes this notebook and the two CSV files are in the same folder.


In [2]:
# Run once if CatBoost is missing from this notebook environment.
%pip install catboost


   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 363.0 kB/s eta 0:04:35
   ---------------------------------------- 0.5/100.2 MB 363.0 kB/s eta 0:04:35
   ---------------------------------------- 0.8/100.2 MB 500.0 kB/s eta 0:03:19
   ---------------------------------------- 1.0/100.2 MB 624.2 kB/s eta 0:02:39
    --------------------------------------- 1.3/100.2 MB 725.3 kB/s eta 0:02:17
    --------------------------------------- 1.8/100.2 MB 874.6 kB/s eta 0:01:53
    -------------------

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix


In [2]:
# Files are next to this notebook. The training CSV was named "train (1).csv".
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "train (1).csv"
TEST_PATH = DATA_DIR / "test.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
TARGET = "satisfaction"
ID_COL = "id"

print("train:", train.shape, "test:", test.shape)
display(train.head())
print("\nTarget distribution:")
print(train[TARGET].value_counts(dropna=False))
print("\nMissing values (training):")
print(train.isna().sum().sort_values(ascending=False).head(10))


train: (699635, 23) test: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False



Target distribution:
satisfaction
False    389296
True     310339
Name: count, dtype: int64

Missing values (training):
Arrival Delay in Minutes      292
id                              0
Leg room service                0
Class                           0
Type of Travel                  0
Customer Type                   0
Gender                          0
Departure Delay in Minutes      0
Cleanliness                     0
Checkin service                 0
dtype: int64


In [3]:
# Separate identifiers and target; do not train on the ID by default.
X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].astype(int)
X_test = test.drop(columns=[ID_COL]).copy()

# CatBoost accepts categorical columns directly; represent missing categories explicitly.
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
for col in cat_features:
    X[col] = X[col].fillna("__MISSING__").astype(str)
    X_test[col] = X_test[col].fillna("__MISSING__").astype(str)

# Ensure numeric columns use consistent numeric values and preserve missing values for CatBoost.
for col in X.columns.difference(cat_features):
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

print("Categorical features:", cat_features)
print("Positive class rate:", y.mean())


Categorical features: ['Gender', 'Customer Type', 'Type of Travel', 'Class']
Positive class rate: 0.44357272006117476


In [7]:
# Fixed stratified holdout gives a first estimate of generalization.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=2500,
    learning_rate=0.07,
    depth=9,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    early_stopping_rounds=150,
    allow_writing_files=False,
    thread_count=-1,
)
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True,
)

valid_proba = model.predict_proba(X_valid)[:, 1]
print(f"Validation ROC AUC: {roc_auc_score(y_valid, valid_proba):.6f}")
print("Best iteration:", model.best_iteration_)


0:	test: 0.9462670	best: 0.9462670 (0)	total: 319ms	remaining: 13m 17s
200:	test: 0.9570692	best: 0.9570692 (200)	total: 56s	remaining: 10m 40s
400:	test: 0.9575977	best: 0.9575986 (399)	total: 1m 56s	remaining: 10m 9s
600:	test: 0.9576826	best: 0.9576911 (595)	total: 3m 15s	remaining: 10m 18s
800:	test: 0.9576619	best: 0.9577116 (739)	total: 4m 33s	remaining: 9m 40s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.9577116236
bestIteration = 739

Shrink model to first 740 iterations.
Validation ROC AUC: 0.957712
Best iteration: 739


## Optional threshold-based check

The competition is scored by ROC AUC, so the probability ranking above is the key result. The next cell is only an optional classification report at threshold 0.5; do not submit these hard labels.


In [5]:
valid_pred = (valid_proba >= 0.5).astype(int)
print(classification_report(y_valid, valid_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_valid, valid_pred))


              precision    recall  f1-score   support

           0     0.9241    0.9425    0.9332     77859
           1     0.9261    0.9028    0.9143     62068

    accuracy                         0.9249    139927
   macro avg     0.9251    0.9227    0.9238    139927
weighted avg     0.9249    0.9249    0.9248    139927

Confusion matrix:
 [[73385  4474]
 [ 6031 56037]]


In [ ]:
# Refit using the full labeled dataset. Use the best holdout iteration (with a safe minimum).
best_iterations = model.best_iteration_
if best_iterations is None or best_iterations <= 0:
    best_iterations = 1500

final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.07,
    depth=9,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1,
)
final_model.fit(X, y, cat_features=cat_features)
test_proba = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_proba})
submission_path = DATA_DIR / "submission_catboost.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path.resolve())
print(submission.head())
print("Rows:", len(submission), "IDs unique:", submission[ID_COL].is_unique)
print("Probabilities in [0, 1]:", submission[TARGET].between(0, 1).all())


0:	total: 253ms	remaining: 7m 10s
200:	total: 46.4s	remaining: 5m 46s
400:	total: 1m 35s	remaining: 5m 11s
600:	total: 2m 25s	remaining: 4m 27s
800:	total: 4m	remaining: 4m 31s
1000:	total: 5m 57s	remaining: 4m 11s
1200:	total: 7m 46s	remaining: 3m 15s
1400:	total: 9m 33s	remaining: 2m 4s
1600:	total: 11m 21s	remaining: 43.9s
1703:	total: 12m 16s	remaining: 0us
Saved: C:\Users\Asus\OneDrive\Desktop\Predicting_Airline_Satisfaction\submission_catboost.csv
       id  satisfaction
0  699635      0.234464
1  699636      0.892555
2  699637      0.911173
3  699638      0.074991
4  699639      0.066961
Rows: 299844 IDs unique: True
Probabilities in [0, 1]: True


## Before submitting

Confirm the final output has exactly the `id,satisfaction` columns and one row for every test ID. Upload `submission_catboost.csv` on the competition's Submit Predictions page. Compare experiments using the same validation strategy; do not choose a model solely because of a small public leaderboard change.
